In [1]:
# ==========================================
# Install Libraries
# ==========================================

!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 98.5 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer

import faiss

import pickle

In [3]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
train_df = pd.read_csv(
    "/content/drive/MyDrive/Fake-News-RAG/datasets/train_clean.csv"
)

train_df.head()

,text,label
0,Says the Annies List political group supports ...,1
1,When did the decline of coal start? It started...,0
2,"Hillary Clinton agrees with John McCain ""by vo...",0
3,Health care reform legislation is likely to ma...,1
4,The economic turnaround started at the end of ...,0


In [5]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!


In [6]:
texts = train_df["text"].tolist()

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(embeddings.shape)

Batches:   0%|          | 0/320 [00:00<?, ?it/s]

(10240, 384)


In [7]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print(index.ntotal)

10240


In [8]:
import os

save_dir = "/content/drive/MyDrive/Fake-News-RAG/vector_db"

os.makedirs(save_dir, exist_ok=True)

faiss.write_index(
    index,
    save_dir + "/faiss_index.bin"
)

print("FAISS index saved!")

FAISS index saved!


In [9]:
with open(
    save_dir + "/documents.pkl",
    "wb"
) as f:

    pickle.dump(
        texts,
        f
    )

print("Documents saved!")

Documents saved!


In [10]:
index = faiss.read_index(
    save_dir + "/faiss_index.bin"
)

with open(
    save_dir + "/documents.pkl",
    "rb"
) as f:

    documents = pickle.load(f)

print("Loaded successfully!")

Loaded successfully!


In [11]:
query = "The government announced free electricity for all citizens."

query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True
)

distances, indices = index.search(
    query_embedding,
    k=5
)

indices

array([[10175,  4833,  7873,  6889,  3547]])

In [12]:
print("User Query:\n")
print(query)

print("\nTop Retrieved News:\n")

for idx in indices[0]:

    print("-" * 80)

    print(documents[idx])

    print()

User Query:

The government announced free electricity for all citizens.

Top Retrieved News:

--------------------------------------------------------------------------------
All non-US citizens, illegal or not, will be provided with free health care services.

--------------------------------------------------------------------------------
Says the federal government is the largest energy user in the country.

--------------------------------------------------------------------------------
We came out against Deepwater and everybody is now paying for [the project] in their electric bills.

--------------------------------------------------------------------------------
On campaign contributions from electric utilities.

--------------------------------------------------------------------------------
We gave every public employee in the state the freedom to choose whether or not they want to be in a union.



In [13]:
from transformers import BertTokenizer
from transformers import BertForSequenceClassification

import torch

In [14]:
model_path = "/content/drive/MyDrive/Fake-News-RAG/models"

tokenizer = BertTokenizer.from_pretrained(
    model_path
)

model = BertForSequenceClassification.from_pretrained(
    model_path
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [17]:
import torch.nn.functional as F

def predict_news(news_text):

    encoding = tokenizer(
        news_text,
        truncation=True,
        padding="max_length",
        max_length=128,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    probabilities = F.softmax(outputs.logits, dim=1)

    confidence = torch.max(probabilities).item()

    prediction = torch.argmax(outputs.logits, dim=1).item()

    label = "Fake" if prediction == 1 else "Real"

    query_embedding = embedding_model.encode(
        [news_text],
        convert_to_numpy=True
    )

    distances, indices = index.search(
        query_embedding,
        k=5
    )

    evidence = []

    for idx in indices[0]:

        evidence.append(documents[idx])

    return label, confidence, evidence

In [19]:
label, confidence, evidence = predict_news(
    "NASA confirms aliens landed in New York yesterday."
)

print("Prediction :", label)
print("Confidence :", round(confidence * 100, 2), "%")

print()

print("Evidence:")

for i, doc in enumerate(evidence, 1):

    print()

    print(f"{i}. {doc}")

Prediction : Fake
Confidence : 63.88 %

Evidence:

1. 134,000 (criminal) aliens have been released by the (Obama) administration in just the past two years.

2. More people in this country have seen UFOs than I think approve of George Bush's presidency.

3. Says these Jersey Shore folks, theyre from New York.

4. Says CIA Director George Tenet told the Bush administration that the Sept. 11, 2001, terrorist attack was coming. So they did have advanced notice.

5. Says Americans invented Pong, Space Invaders and the iPhone.
